# Setup
Run the install cell once, then restart the kernel and run all the cells.

In [1]:
%pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
%pip install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 bitsandbytes==0.44.1 torch_geometric==2.6.1
%pip install scikit-learn==1.5.2 scipy==1.14.1 pandas==2.2.3 matplotlib==3.9.2 opencv-python==4.10.0.84 sentencepiece==0.2.0 numpy==2.2.6 Pillow==10.4.0 requests==2.32.3 huggingface_hub==0.26.5

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, io, json, zipfile, platform, requests
import torch, diffusers, transformers, torch_geometric

gpu = torch.cuda.get_device_properties(0)
env = {}
env["python"] = sys.version.split()[0]
env["os"] = platform.platform()
env["torch"] = torch.__version__
env["diffusers"] = diffusers.__version__
env["transformers"] = transformers.__version__
env["torch_geometric"] = torch_geometric.__version__
env["gpu"] = gpu.name
env["gpu_memory_gb"] = round(gpu.total_memory/1024**3,1)

os.makedirs("results",exist_ok=True)
with open("results/env.json","w") as f:
    json.dump(env,f,indent=2)
print(env)

{'python': '3.12.14', 'os': 'Windows-11-10.0.26200-SP0', 'torch': '2.5.1+cu124', 'diffusers': '0.31.0', 'transformers': '4.46.3', 'torch_geometric': '2.6.1', 'gpu': 'NVIDIA GeForce RTX 4060 Laptop GPU', 'gpu_memory_gb': 8.0}


## Data
DreamBooth images and the Amazon fraud graph

In [3]:
subjects = ["berry_bowl", "candle", "clock", "fancy_boot", "shiny_sneaker", "vase", "can", "rc_car",
            "teapot", "backpack", "colorful_sneaker", "bear_plushie"]

if not os.path.exists("data/dreambooth/bear_plushie"):
    r = requests.get("https://codeload.github.com/google/dreambooth/zip/refs/heads/main")
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for name in z.namelist():
        parts = name.split("/")
        if len(parts)==4 and parts[2] in subjects and parts[3]!="":
            folder = "data/dreambooth/"+parts[2]
            os.makedirs(folder,exist_ok=True)
            with open(folder+"/"+parts[3],"wb") as f:
                f.write(z.read(name))

if not os.path.exists("data/amazon/Amazon.mat"):
    r = requests.get("https://github.com/YingtongDou/CARE-GNN/raw/master/data/Amazon.zip")
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("data/amazon")

for s in subjects:
    print(s,os.listdir("data/dreambooth/"+s))
print(os.listdir("data/amazon"))

berry_bowl ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
candle ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg']
clock ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
fancy_boot ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
shiny_sneaker ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
vase ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
can ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
rc_car ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg']
teapot ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg']
backpack ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg', '05.jpg']
colorful_sneaker ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg']
bear_plushie ['00.jpg', '01.jpg', '02.jpg', '03.jpg', '04.jpg']
['Amazon.mat']


In [4]:
from huggingface_hub import model_info

models = ["stable-diffusion-v1-5/stable-diffusion-v1-5", "lllyasviel/sd-controlnet-canny", "timbrooks/instruct-pix2pix",
          "h94/IP-Adapter", "openai/clip-vit-large-patch14", "facebook/dino-vits16", "facebook/dino-vitb8",
          "Qwen/Qwen2.5-1.5B-Instruct", "microsoft/Phi-3.5-mini-instruct"]

if not os.path.exists("data/model_versions.json"):
    versions = {}
    for m in models:
        versions[m] = model_info(m).sha
    with open("data/model_versions.json","w") as f:
        json.dump(versions,f,indent=2)

with open("data/model_versions.json") as f:
    print(f.read())

{
  "stable-diffusion-v1-5/stable-diffusion-v1-5": "451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
  "lllyasviel/sd-controlnet-canny": "7f2f69197050967007f6bbd23ab5e52f0384162a",
  "timbrooks/instruct-pix2pix": "31519b5cb02a7fd89b906d88731cd4d6a7bbf88d",
  "h94/IP-Adapter": "018e402774aeeddd60609b4ecdb7e298259dc729",
  "openai/clip-vit-large-patch14": "32bd64288804d66eefd0ccbe215aa642df71cc41",
  "facebook/dino-vits16": "abe3b354cb6a9b6f146096b14a4a9d7eecbcb4bd",
  "facebook/dino-vitb8": "f1ca762fe312f2617c0f37f3d56a5af83847bf6c",
  "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "microsoft/Phi-3.5-mini-instruct": "2fe192450127e6a83f7441aef6e3ca586c338b77"
}
